# 2-Layer Position Decoding Analysis

This notebook analyzes how the second layer (MLP or attention) can decode position from the uniform attention patterns created by block 0.

## Key Questions:
1. **Block 0 Output**: What position signal exists after uniform attention averaging?
2. **Second Attention (Block 1)**: Does block 1's attention help decode position?
3. **MLP Decoding**: How does the first MLP transform the averaged embeddings into position-informative features?
4. **Decoding Vector Alignment**: Does the MLP align with the theoretical decoding vector?

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add nanoGPT to path
sys.path.insert(0, str(Path.cwd().parent / "nanoGPT"))
from model_position_classifier import GPTPositionClassifier, GPTPositionClassifierConfig

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## Load 2-Layer Model

In [ ]:
def load_checkpoint(ckpt_path, device='cuda'):
    """Load a position-regression checkpoint."""
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model_args = checkpoint.get('model_args', {})
    
    config = GPTPositionClassifierConfig(
        n_layer=model_args.get('n_layer', 2),
        n_head=model_args.get('n_head', 12),
        n_embd=model_args.get('n_embd', 768),
        block_size=model_args.get('block_size', 128),
        vocab_size=model_args.get('vocab_size', 50304),
        dropout=0.0,
        bias=model_args.get('bias', False),
        use_positional_embedding=False,
        norm_type=model_args.get('norm_type', 'layernorm'),
        use_regression=True,
        compute_lm_loss=model_args.get('compute_lm_loss', False),
    )
    
    model = GPTPositionClassifier(config)
    state_dict = checkpoint['model']
    
    # Handle compiled models
    unwrapped = {}
    for key, value in state_dict.items():
        clean_key = key[10:] if key.startswith('_orig_mod.') else key
        unwrapped[clean_key] = value
    
    model.load_state_dict(unwrapped, strict=False)
    model.to(device)
    model.eval()
    
    return model, checkpoint

# Load latest checkpoint
ckpt_path = Path.cwd().parent / "nanoGPT" / "out-posreg-2layer-until-mlp" / "ckpt.pt"
print(f"Loading checkpoint: {ckpt_path}")
model, checkpoint = load_checkpoint(ckpt_path, device)

print(f"\nModel config:")
print(f"  n_layer: {model.config.n_layer}")
print(f"  n_head: {model.config.n_head}")
print(f"  n_embd: {model.config.n_embd}")
print(f"  block_size: {model.config.block_size}")
print(f"\nTraining info:")
print(f"  iter_num: {checkpoint.get('iter_num', 'N/A')}")
print(f"  best_val_loss: {checkpoint.get('best_val_loss', 'N/A'):.4f}")

## Generate Sample Data

In [ ]:
# Generate random token sequences
n_samples = 1000
context_length = 128
vocab_size = model.config.vocab_size

torch.manual_seed(42)
tokens = torch.randint(0, vocab_size, (n_samples, context_length), device=device)

print(f"Generated {n_samples} sequences of length {context_length}")
print(f"Token range: [{tokens.min()}, {tokens.max()}]")

## Extract Activations at Each Layer

We'll hook into the model to capture:
- Embeddings
- Block 0: post-LN1, post-attention, post-LN2, post-MLP
- Block 1: post-LN1, post-attention, post-LN2, post-MLP (final)

In [ ]:
def extract_activations(model, tokens):
    """Extract activations at each critical point in the 2-layer model."""
    activations = {}
    
    def make_hook(name):
        def hook(module, input, output):
            if isinstance(output, tuple):
                activations[name] = output[0].detach().cpu()
            else:
                activations[name] = output.detach().cpu()
        return hook
    
    hooks = []
    
    # Embeddings
    hooks.append(model.transformer.wte.register_forward_hook(make_hook('embed')))
    
    # Block 0
    hooks.append(model.transformer.h[0].ln_1.register_forward_hook(make_hook('block0_post_ln1')))
    hooks.append(model.transformer.h[0].attn.register_forward_hook(make_hook('block0_post_attn')))
    hooks.append(model.transformer.h[0].ln_2.register_forward_hook(make_hook('block0_post_ln2')))
    hooks.append(model.transformer.h[0].mlp.register_forward_hook(make_hook('block0_post_mlp')))
    
    # Block 1
    hooks.append(model.transformer.h[1].ln_1.register_forward_hook(make_hook('block1_post_ln1')))
    hooks.append(model.transformer.h[1].attn.register_forward_hook(make_hook('block1_post_attn')))
    hooks.append(model.transformer.h[1].ln_2.register_forward_hook(make_hook('block1_post_ln2')))
    hooks.append(model.transformer.h[1].mlp.register_forward_hook(make_hook('block1_post_mlp')))
    
    # Forward pass
    with torch.no_grad():
        _ = model(tokens)
    
    # Remove hooks
    for hook in hooks:
        hook.remove()
    
    # Also get residual stream at key points
    with torch.no_grad():
        x = model.transformer.wte(tokens)
        
        # After block 0 (with residual)
        h0 = model.transformer.h[0]
        x = x + h0.attn(h0.ln_1(x))[0]
        activations['block0_residual_post_attn'] = x.detach().cpu()
        x = x + h0.mlp(h0.ln_2(x))
        activations['block0_residual_post_mlp'] = x.detach().cpu()
        
        # After block 1 (with residual)
        h1 = model.transformer.h[1]
        x = x + h1.attn(h1.ln_1(x))[0]
        activations['block1_residual_post_attn'] = x.detach().cpu()
        x = x + h1.mlp(h1.ln_2(x))
        activations['block1_residual_post_mlp'] = x.detach().cpu()
    
    return activations

print("Extracting activations...")
activations = extract_activations(model, tokens)
print(f"Extracted {len(activations)} activation tensors")
for name, tensor in activations.items():
    print(f"  {name}: {tensor.shape}")

## 1. Analyze Block 0 Output: Uniform Attention Averaging

Block 0's uniform attention creates position-dependent variance through averaging.

In [ ]:
# Analyze attention patterns in block 0
def get_attention_pattern(model, tokens, layer_idx=0, head_idx=0):
    """Extract attention pattern from a specific head."""
    attn_weights = []
    
    def hook(module, input, output):
        # output is (attn_output, attn_weights)
        if isinstance(output, tuple) and len(output) > 1:
            attn_weights.append(output[1].detach().cpu())
    
    handle = model.transformer.h[layer_idx].attn.register_forward_hook(hook)
    
    with torch.no_grad():
        _ = model(tokens[:10])  # Just 10 samples for visualization
    
    handle.remove()
    
    if attn_weights:
        return attn_weights[0]
    return None

# Get attention pattern for first head of block 0
attn_pattern = get_attention_pattern(model, tokens, layer_idx=0, head_idx=0)

if attn_pattern is not None:
    # Plot attention pattern for first sample
    sample_idx = 0
    attn_sample = attn_pattern[sample_idx, 0, :, :].numpy()  # First head
    
    fig = px.imshow(
        attn_sample,
        labels=dict(x="Key Position", y="Query Position", color="Attention Weight"),
        title="Block 0, Head 0: Attention Pattern (Uniform Averaging)",
        color_continuous_scale="Viridis",
    )
    fig.update_layout(width=700, height=600)
    fig.show()
    
    # Check uniformity: compute variance along each query row
    row_variances = np.var(attn_sample, axis=1)
    print(f"\nAttention uniformity check:")
    print(f"  Mean row variance: {row_variances.mean():.6f}")
    print(f"  Max row variance: {row_variances.max():.6f}")
    print(f"  Expected for uniform: ~0 (closer to 0 = more uniform)")

In [ ]:
# Analyze variance decay after block 0 attention
def analyze_variance_by_position(activations_tensor, layer_name):
    """Compute variance of activations at each position."""
    # activations_tensor: [batch, seq_len, d_model]
    variances = activations_tensor.var(dim=-1).mean(dim=0).numpy()  # Average across batch
    positions = np.arange(len(variances))
    
    # Theoretical variance: 1/(i+1) for uniform averaging
    theoretical = 1.0 / (positions + 1)
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=positions, y=variances, mode='lines+markers', name='Observed'))
    fig.add_trace(go.Scatter(x=positions, y=theoretical, mode='lines', name='Theoretical 1/(i+1)', line=dict(dash='dash')))
    fig.update_layout(
        title=f"Variance Decay by Position: {layer_name}",
        xaxis_title="Position",
        yaxis_title="Variance",
        width=900,
        height=500,
    )
    fig.show()
    
    # Correlation with theory
    corr = np.corrcoef(variances, theoretical)[0, 1]
    print(f"Correlation with 1/(i+1): {corr:.4f}")
    
    return variances

print("\n=== Block 0: Post-Attention (before residual) ===")
var_block0_attn = analyze_variance_by_position(
    activations['block0_post_attn'], 'Block 0 Post-Attention'
)

print("\n=== Block 0: Residual Stream Post-Attention ===")
var_block0_res_attn = analyze_variance_by_position(
    activations['block0_residual_post_attn'], 'Block 0 Residual Post-Attention'
)

## 2. Position Probing at Each Layer

Train linear probes to predict position from activations at each layer.

In [ ]:
def probe_position(activations_tensor, positions, test_size=0.2):
    """Train a linear probe to predict position from activations."""
    # activations_tensor: [batch, seq_len, d_model]
    # Flatten batch and sequence dimensions
    X = activations_tensor.reshape(-1, activations_tensor.shape[-1]).numpy()
    y = positions.flatten().numpy()
    
    # Split train/test
    n_train = int(len(X) * (1 - test_size))
    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]
    
    # Train probe
    probe = LinearRegression()
    probe.fit(X_train, y_train)
    
    # Evaluate
    train_r2 = probe.score(X_train, y_train)
    test_r2 = probe.score(X_test, y_test)
    
    return {
        'train_r2': train_r2,
        'test_r2': test_r2,
        'probe': probe,
    }

# Create position labels
positions = torch.arange(context_length).unsqueeze(0).expand(n_samples, -1)

# Probe each layer
layers_to_probe = [
    'embed',
    'block0_post_ln1',
    'block0_residual_post_attn',
    'block0_post_ln2',
    'block0_residual_post_mlp',
    'block1_post_ln1',
    'block1_residual_post_attn',
    'block1_post_ln2',
    'block1_residual_post_mlp',
]

probe_results = {}
for layer_name in layers_to_probe:
    if layer_name in activations:
        result = probe_position(activations[layer_name], positions)
        probe_results[layer_name] = result
        print(f"{layer_name:30s} - Train R²: {result['train_r2']:.4f}, Test R²: {result['test_r2']:.4f}")

In [ ]:
# Visualize probe performance
layer_names = list(probe_results.keys())
test_r2_values = [probe_results[name]['test_r2'] for name in layer_names]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=layer_names,
    y=test_r2_values,
    marker_color='steelblue',
))
fig.update_layout(
    title="Position Probe R² Across Layers (2-Layer Model)",
    xaxis_title="Layer",
    yaxis_title="Test R²",
    xaxis_tickangle=-45,
    width=1000,
    height=500,
)
fig.show()

print("\n=== Key Observations ===")
print(f"Embedding R²: {probe_results['embed']['test_r2']:.4f} (should be ~0, no position info)")
print(f"Block 0 after attention: {probe_results['block0_residual_post_attn']['test_r2']:.4f}")
print(f"Block 0 after MLP: {probe_results['block0_residual_post_mlp']['test_r2']:.4f}")
print(f"Block 1 after attention: {probe_results['block1_residual_post_attn']['test_r2']:.4f}")
print(f"Block 1 after MLP (final): {probe_results['block1_residual_post_mlp']['test_r2']:.4f}")

## 3. Norm-Position Correlation Analysis

Check if activation norm correlates with position at each layer.

In [ ]:
def compute_norm_position_correlation(activations_tensor, positions):
    """Compute correlation between activation norm and position."""
    # activations_tensor: [batch, seq_len, d_model]
    norms = torch.norm(activations_tensor, dim=-1).flatten().numpy()
    pos = positions.flatten().numpy()
    
    corr = np.corrcoef(norms, pos)[0, 1]
    return corr

norm_correlations = {}
for layer_name in layers_to_probe:
    if layer_name in activations:
        corr = compute_norm_position_correlation(activations[layer_name], positions)
        norm_correlations[layer_name] = corr
        print(f"{layer_name:30s} - Norm-Position Correlation: {corr:+.4f}")

In [ ]:
# Visualize norm-position correlation
fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(norm_correlations.keys()),
    y=list(norm_correlations.values()),
    marker_color=['green' if v > 0 else 'red' for v in norm_correlations.values()],
))
fig.update_layout(
    title="Norm-Position Correlation Across Layers",
    xaxis_title="Layer",
    yaxis_title="Correlation",
    xaxis_tickangle=-45,
    width=1000,
    height=500,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.show()

## 4. Decoding Vector Analysis

Compute the theoretical decoding vector and check alignment with MLP weights.

In [ ]:
def compute_decoding_vector(model, layer_idx=0):
    """Compute decoding vector from W_V and W_O at specified layer."""
    block = model.transformer.h[layer_idx]
    embeddings = model.transformer.wte.weight.detach()
    
    # Apply LN1 to embeddings
    with torch.no_grad():
        ln_embeddings = block.ln_1(embeddings)
    
    # Sum over all tokens
    summed = ln_embeddings.sum(dim=0)  # [d_model]
    
    # Extract W_V and W_O
    n_embd = model.config.n_embd
    c_attn_weight = block.attn.c_attn.weight  # [3*d_model, d_model]
    w_v = c_attn_weight[2 * n_embd:, :]  # Value weights
    w_o = block.attn.c_proj.weight  # Output projection
    
    # Decoding vector
    decoding_v = summed @ w_v.T  # [d_model]
    decoding_v_o = decoding_v @ w_o.T  # After output projection
    
    return {
        'summed_embeddings': summed.cpu(),
        'decoding_v': decoding_v.cpu(),
        'decoding_v_o': decoding_v_o.cpu(),
    }

# Compute decoding vectors for both blocks
decoding_block0 = compute_decoding_vector(model, layer_idx=0)
decoding_block1 = compute_decoding_vector(model, layer_idx=1)

print("Decoding vectors computed:")
print(f"  Block 0 - decoding_v shape: {decoding_block0['decoding_v'].shape}")
print(f"  Block 0 - decoding_v_o shape: {decoding_block0['decoding_v_o'].shape}")
print(f"  Block 1 - decoding_v shape: {decoding_block1['decoding_v'].shape}")
print(f"  Block 1 - decoding_v_o shape: {decoding_block1['decoding_v_o'].shape}")

In [ ]:
def cosine_similarity_to_rows(vector, matrix):
    """Compute cosine similarity between vector and each row of matrix."""
    vec_norm = torch.norm(vector) + 1e-8
    row_norms = torch.norm(matrix, dim=1) + 1e-8
    sims = (matrix @ vector) / (row_norms * vec_norm)
    return sims.numpy()

def cosine_similarity_to_cols(vector, matrix):
    """Compute cosine similarity between vector and each column of matrix."""
    return cosine_similarity_to_rows(vector, matrix.T)

# Analyze Block 0 MLP alignment
block0_mlp = model.transformer.h[0].mlp
c_fc_weight = block0_mlp.c_fc.weight.detach().cpu()  # [4*d_model, d_model]
c_proj_weight = block0_mlp.c_proj.weight.detach().cpu()  # [d_model, 4*d_model]

# Alignment with c_fc (input -> hidden)
fc_sims_v = cosine_similarity_to_rows(decoding_block0['decoding_v'], c_fc_weight)
fc_sims_vo = cosine_similarity_to_rows(decoding_block0['decoding_v_o'], c_fc_weight)

# Alignment with c_proj (hidden -> output) - check columns
proj_sims_v = cosine_similarity_to_cols(decoding_block0['decoding_v'], c_proj_weight)
proj_sims_vo = cosine_similarity_to_cols(decoding_block0['decoding_v_o'], c_proj_weight)

print("\n=== Block 0 MLP Alignment ===")
print(f"\nDecoding vector (W_V · sum(E)) vs c_fc rows:")
print(f"  Max |cos|: {np.abs(fc_sims_v).max():.4f}")
print(f"  Mean |cos|: {np.abs(fc_sims_v).mean():.4f}")
print(f"  Top-10 mean |cos|: {np.sort(np.abs(fc_sims_v))[-10:].mean():.4f}")

print(f"\nDecoding vector (W_O W_V · sum(E)) vs c_fc rows:")
print(f"  Max |cos|: {np.abs(fc_sims_vo).max():.4f}")
print(f"  Mean |cos|: {np.abs(fc_sims_vo).mean():.4f}")
print(f"  Top-10 mean |cos|: {np.sort(np.abs(fc_sims_vo))[-10:].mean():.4f}")

In [ ]:
# Visualize alignment distribution
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'W_V · sum(E) vs c_fc rows',
        'W_O W_V · sum(E) vs c_fc rows',
        'W_V · sum(E) vs c_proj cols',
        'W_O W_V · sum(E) vs c_proj cols',
    )
)

fig.add_trace(
    go.Histogram(x=fc_sims_v, nbinsx=50, name='W_V · sum(E)'),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=fc_sims_vo, nbinsx=50, name='W_O W_V · sum(E)'),
    row=1, col=2
)
fig.add_trace(
    go.Histogram(x=proj_sims_v, nbinsx=50, name='W_V · sum(E)'),
    row=2, col=1
)
fig.add_trace(
    go.Histogram(x=proj_sims_vo, nbinsx=50, name='W_O W_V · sum(E)'),
    row=2, col=2
)

fig.update_layout(
    title_text="Block 0: Decoding Vector Alignment with MLP Weights",
    showlegend=False,
    width=1000,
    height=700,
)
fig.update_xaxes(title_text="Cosine Similarity")
fig.update_yaxes(title_text="Count")
fig.show()

## 5. MLP Neuron Activation Analysis

Check which MLP neurons activate strongly for different positions.

In [ ]:
def extract_mlp_hidden_activations(model, tokens, layer_idx=0):
    """Extract MLP hidden layer activations (after GELU)."""
    hidden_acts = []
    
    def hook(module, input, output):
        # After c_fc and GELU
        hidden_acts.append(input[0].detach().cpu())  # Input to c_proj is output of GELU
    
    handle = model.transformer.h[layer_idx].mlp.c_proj.register_forward_hook(hook)
    
    with torch.no_grad():
        _ = model(tokens)
    
    handle.remove()
    
    return hidden_acts[0] if hidden_acts else None

# Extract MLP hidden activations for block 0
mlp_hidden_block0 = extract_mlp_hidden_activations(model, tokens, layer_idx=0)
print(f"Block 0 MLP hidden activations shape: {mlp_hidden_block0.shape}")

# Compute position-neuron correlation
# mlp_hidden_block0: [batch, seq_len, 4*d_model]
n_neurons = mlp_hidden_block0.shape[-1]
neuron_position_corrs = []

for neuron_idx in range(n_neurons):
    neuron_acts = mlp_hidden_block0[:, :, neuron_idx].flatten().numpy()
    pos = positions.flatten().numpy()
    corr = np.corrcoef(neuron_acts, pos)[0, 1]
    neuron_position_corrs.append(corr)

neuron_position_corrs = np.array(neuron_position_corrs)

print(f"\nNeuron-position correlations:")
print(f"  Mean |corr|: {np.abs(neuron_position_corrs).mean():.4f}")
print(f"  Max |corr|: {np.abs(neuron_position_corrs).max():.4f}")
print(f"  Top-10 mean |corr|: {np.sort(np.abs(neuron_position_corrs))[-10:].mean():.4f}")

# Find top position-correlated neurons
top_neurons = np.argsort(np.abs(neuron_position_corrs))[-20:]
print(f"\nTop 20 position-correlated neurons:")
for idx in top_neurons[::-1]:
    print(f"  Neuron {idx:4d}: corr = {neuron_position_corrs[idx]:+.4f}")

In [ ]:
# Visualize neuron-position correlation distribution
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=neuron_position_corrs,
    nbinsx=100,
    marker_color='steelblue',
))
fig.update_layout(
    title="Block 0 MLP: Neuron-Position Correlation Distribution",
    xaxis_title="Correlation",
    yaxis_title="Count",
    width=900,
    height=500,
)
fig.show()

# Show top neurons' activations across positions
top_3_neurons = np.argsort(np.abs(neuron_position_corrs))[-3:]

fig = go.Figure()
for neuron_idx in top_3_neurons[::-1]:
    # Average activation at each position across batch
    avg_acts = mlp_hidden_block0[:, :, neuron_idx].mean(dim=0).numpy()
    fig.add_trace(go.Scatter(
        x=np.arange(context_length),
        y=avg_acts,
        mode='lines+markers',
        name=f'Neuron {neuron_idx} (r={neuron_position_corrs[neuron_idx]:+.3f})',
    ))

fig.update_layout(
    title="Top 3 Position-Correlated Neurons: Activation vs Position",
    xaxis_title="Position",
    yaxis_title="Mean Activation",
    width=900,
    height=500,
)
fig.show()

## 6. Block 1 Attention: Does it Help?

Analyze whether block 1's attention contributes to position decoding or if the MLP does all the work.

In [ ]:
# Compare position probing before and after block 1 attention
r2_before_block1_attn = probe_results['block1_post_ln1']['test_r2']
r2_after_block1_attn = probe_results['block1_residual_post_attn']['test_r2']
r2_after_block1_mlp = probe_results['block1_residual_post_mlp']['test_r2']

print("\n=== Block 1 Contribution ===")
print(f"Before block 1 attention: R² = {r2_before_block1_attn:.4f}")
print(f"After block 1 attention:  R² = {r2_after_block1_attn:.4f}")
print(f"After block 1 MLP:        R² = {r2_after_block1_mlp:.4f}")
print(f"\nAttention contribution: {r2_after_block1_attn - r2_before_block1_attn:+.4f}")
print(f"MLP contribution:       {r2_after_block1_mlp - r2_after_block1_attn:+.4f}")

# Visualize
fig = go.Figure()
stages = ['Before Block 1', 'After Attn', 'After MLP']
r2_values = [r2_before_block1_attn, r2_after_block1_attn, r2_after_block1_mlp]

fig.add_trace(go.Bar(
    x=stages,
    y=r2_values,
    marker_color=['lightblue', 'steelblue', 'darkblue'],
))
fig.update_layout(
    title="Position Decoding Through Block 1",
    yaxis_title="Test R²",
    width=700,
    height=500,
)
fig.show()

## 7. Direction vs Norm: Where is Position Encoded?

Decompose activations into direction and norm to understand encoding mechanism.

In [ ]:
def probe_direction_and_norm(activations_tensor, positions):
    """Probe position from direction (unit vector) and norm separately."""
    # activations_tensor: [batch, seq_len, d_model]
    norms = torch.norm(activations_tensor, dim=-1, keepdim=True)
    directions = activations_tensor / (norms + 1e-8)
    
    # Flatten
    X_full = activations_tensor.reshape(-1, activations_tensor.shape[-1]).numpy()
    X_direction = directions.reshape(-1, directions.shape[-1]).numpy()
    X_norm = norms.reshape(-1, 1).numpy()
    y = positions.flatten().numpy()
    
    # Probe
    probe_full = LinearRegression().fit(X_full, y)
    probe_direction = LinearRegression().fit(X_direction, y)
    probe_norm = LinearRegression().fit(X_norm, y)
    
    return {
        'full_r2': probe_full.score(X_full, y),
        'direction_r2': probe_direction.score(X_direction, y),
        'norm_r2': probe_norm.score(X_norm, y),
    }

# Analyze key layers
key_layers = [
    'block0_residual_post_attn',
    'block0_post_ln2',
    'block0_residual_post_mlp',
    'block1_residual_post_attn',
    'block1_post_ln2',
    'block1_residual_post_mlp',
]

direction_norm_results = {}
for layer_name in key_layers:
    if layer_name in activations:
        result = probe_direction_and_norm(activations[layer_name], positions)
        direction_norm_results[layer_name] = result
        print(f"{layer_name:30s} - Full: {result['full_r2']:.4f}, Dir: {result['direction_r2']:.4f}, Norm: {result['norm_r2']:.4f}")

In [ ]:
# Visualize direction vs norm encoding
layers = list(direction_norm_results.keys())
full_r2 = [direction_norm_results[l]['full_r2'] for l in layers]
dir_r2 = [direction_norm_results[l]['direction_r2'] for l in layers]
norm_r2 = [direction_norm_results[l]['norm_r2'] for l in layers]

fig = go.Figure()
fig.add_trace(go.Bar(x=layers, y=full_r2, name='Full Vector', marker_color='steelblue'))
fig.add_trace(go.Bar(x=layers, y=dir_r2, name='Direction Only', marker_color='orange'))
fig.add_trace(go.Bar(x=layers, y=norm_r2, name='Norm Only', marker_color='green'))

fig.update_layout(
    title="Position Encoding: Direction vs Norm Across Layers",
    xaxis_title="Layer",
    yaxis_title="R²",
    barmode='group',
    xaxis_tickangle=-45,
    width=1100,
    height=600,
    legend=dict(x=0.7, y=0.95),
)
fig.show()

## Summary & Conclusions

Run all analyses above to understand:

1. **Block 0 creates position signal** via uniform attention averaging (variance ∝ 1/(i+1))
2. **Block 0 MLP** transforms this into stronger position signal
3. **Block 1 attention** may or may not contribute (check R² delta)
4. **Block 1 MLP** finalizes position decoding for regression head
5. **Decoding vector alignment** shows if MLPs learn the theoretical optimal direction
6. **Direction vs Norm** reveals encoding mechanism (pre-LN vs post-LN)

### Key Metrics to Watch:
- Probe R² progression through layers
- MLP neuron-position correlations
- Decoding vector alignment (|cos| with MLP weights)
- Norm-position correlation sign flips after LayerNorm